In [1]:
import json
import os
import time
from ollama import Client

# --- CONFIGURAÇÕES ---
ARQUIVO_ENTRADA = "estrutura_livro.json"
ARQUIVO_SAIDA = "estrutura_livro_traduzido.json"
MODELO_OLLAMA = "translategemma:12b"  # Modifique para a tag exata que você tem instalada

# Inicializa o cliente do Ollama (padrão local: http://localhost:11434)
client = Client()

def carregar_dados(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)

def salvar_dados(dados, caminho):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=4)

def traduzir_texto(texto_original):
    """Envia o texto com tags HTML para o Gemma 3 e exige que a estrutura seja mantida."""
    prompt = (
        "You are an expert literary translator from English to Portuguese.\n"
        "Translate the following text to Portuguese. Rules:\n"
        "1. Maintain the exact same HTML tags (like <i>, <b>, <p>) in their correct corresponding positions.\n"
        "2. Do not add any commentary, explanations, or notes. Output ONLY the translated text.\n"
        "3. Preserve the tone and style of the original book.\n\n"
        f"Text to translate:\n{texto_original}"
    )
    
    try:
        response = client.generate(
            model=MODELO_OLLAMA,
            prompt=prompt,
            options={
                "temperature": 0.3, # Baixo para manter a tradução factual e precisa
                "top_p": 0.9
            }
        )
        return response['response'].strip()
    except Exception as e:
        print(f"\n[ERRO] Falha ao comunicar com o Ollama: {e}")
        return None

def pipeline_traducao():
    # 1. Carrega ou cria o arquivo de saída (para checkpoint)
    if os.path.exists(ARQUIVO_SAIDA):
        print("   -> Arquivo de tradução existente encontrado. Retomando progresso...")
        dados_traduzidos = carregar_dados(ARQUIVO_SAIDA)
    else:
        print("   -> Criando novo arquivo de tradução...")
        dados_originais = carregar_dados(ARQUIVO_ENTRADA)
        # Clona a estrutura básica, mas sem os capítulos ainda
        dados_traduzidos = {
            "titulo_arquivo": dados_originais["titulo_arquivo"],
            "capitulos": []
        }
        salvar_dados(dados_traduzidos, ARQUIVO_SAIDA)

    # Carrega os dados originais para comparar o que falta fazer
    dados_originais = carregar_dados(ARQUIVO_ENTRADA)
    
    # Mapeia quais capítulos já foram 100% traduzidos e salvos anteriormente
    caps_concluidos = {cap["id_capitulo"] for cap in dados_traduzidos["capitulos"]}

    print(f"Iniciando tradução do livro com o modelo {MODELO_OLLAMA}...\n")

    for cap_original in dados_originais["capitulos"]:
        id_cap = cap_original["id_capitulo"]
        
        # Pula se o capítulo já foi traduzido em uma execução anterior
        if id_cap in caps_concluidos:
            print(f"[PULADO] Capítulo {id_cap}: '{cap_original['titulo_capitulo']}' já traduzido.")
            continue
            
        print(f"[PROCESSANDO] Capítulo {id_cap}: '{cap_original['titulo_capitulo']}'")
        
        # Cria a estrutura do novo capítulo
        novo_capitulo = {
            "id_capitulo": id_cap,
            "nome_arquivo_interno": cap_original["nome_arquivo_interno"],
            "titulo_capitulo": cap_original["titulo_capitulo"], # Geralmente títulos curtos traduzimos depois ou via prompt
            "conteudo": []
        }
        
        total_elementos = len(cap_original["conteudo"])
        
        for idx, elemento in enumerate(cap_original["conteudo"], 1):
            # Se for imagem, apenas copia sem gastar IA
            if elemento["tipo"] == "imagem":
                novo_capitulo["conteudo"].append(elemento)
                continue
                
            # Se for texto, envia para o Ollama
            if elemento["tipo"] == "texto":
                print(f"  -> Traduzindo bloco {idx}/{total_elementos} (ID: {elemento['id']})...", end="\r")
                
                texto_traduzido = traduzir_texto(elemento["original"])
                
                # Se falhar catastroficamente (Ollama caiu), salva o que deu e fecha o script com segurança
                if texto_traduzido is None:
                    print(f"\n[AVISO] Pipeline pausada devido a erro no bloco {elemento['id']}. Salvando progresso seguro...")
                    return
                
                # Monta o bloco traduzido
                bloco_traduzido = elemento.copy()
                bloco_traduzido["traduzido"] = texto_traduzido
                novo_capitulo["conteudo"].append(bloco_traduzido)
                
                # Pequena folga para o processador/GPU respirar entre blocos (opcional)
                time.sleep(0.1)

        # --- SALVAMENTO EM INTERVALOS (Fim de cada Capítulo) ---
        dados_traduzidos["capitulos"].append(novo_capitulo)
        salvar_dados(dados_traduzidos, ARQUIVO_SAIDA)
        print(f"\n[CHECKPOINT] Capítulo {id_cap} salvo com sucesso no JSON!\n")

    print("🎉 Parabéns! O livro completo foi traduzido offline de forma segura!")

if __name__ == "__main__":
    pipeline_traducao()

   -> Arquivo de tradução existente encontrado. Retomando progresso...
Iniciando tradução do livro com o modelo translategemma:12b...

[PULADO] Capítulo 1: 'Capítulo 1' já traduzido.
[PULADO] Capítulo 2: 'From Publishers Weekly' já traduzido.
[PULADO] Capítulo 3: 'Capítulo 3' já traduzido.
[PULADO] Capítulo 4: 'Capítulo 4' já traduzido.
[PULADO] Capítulo 5: 'Capítulo 5' já traduzido.
[PULADO] Capítulo 6: 'Capítulo 6' já traduzido.
[PULADO] Capítulo 7: 'Capítulo 7' já traduzido.
[PULADO] Capítulo 8: 'Capítulo 8' já traduzido.
[PULADO] Capítulo 9: 'Capítulo 9' já traduzido.
[PULADO] Capítulo 10: 'Capítulo 10' já traduzido.
[PULADO] Capítulo 11: 'Capítulo 11' já traduzido.
[PULADO] Capítulo 12: 'Capítulo 12' já traduzido.
[PULADO] Capítulo 13: 'Capítulo 13' já traduzido.
[PULADO] Capítulo 14: 'Capítulo 14' já traduzido.
[PULADO] Capítulo 15: 'Capítulo 15' já traduzido.
[PULADO] Capítulo 16: 'Capítulo 16' já traduzido.
[PULADO] Capítulo 17: 'Capítulo 17' já traduzido.
[PULADO] Capítulo 18: